In [ ]:
# ==============================================================
# C1_bedrock.ipynb
# LICENSE:
# SPDX-License-Identifier: MIT
# Copyright (c)  2026 Natalie Sokalska
#
#
# Computes a time-averaged bedrock elevation using the
# Nye (1952) perfect-plasticity method.
#
# PROBLEM THIS SOLVES:
#   A Nye bedrock derived from a single year's DEM is calibrated
#   to that year's surface slope. Using it with a different year's
#   surface gives inconsistent ice thickness because the surface
#   slope changes as the glacier thins over time. This script fixes
#   that by computing the Nye bedrock from a mean surface averaged
#   across all available survey years.
#
# WHAT IT DOES:
#   1. Loads each year's DEM and glacier outline mask
#   2. For each mesh node, averages only the years when that node
#      was inside the glacier 
#   3. Runs the Nye thickness calculation and Tikhonov (J_h)
#      smoothing pipeline
#   4. Saves the resulting bedrock to an HDF5 checkpoint file
#
# ==============================================================

import firedrake
import icepack
import icepack.meshing
import geojson
import rasterio
import numpy as np
from scipy.interpolate import NearestNDInterpolator
from shapely.geometry import shape, Point   
from shapely.ops import unary_union


# ==============================================================
# CONFIGURATION
# Edit this section to match your dataset.
# ==============================================================

# survey year (DSM), glacier outline that will be used for that years computation
# if there is no outline data - use the nearest available outline 
# set this for your glacier
YEARS = {
    # year: ("path/to/dem_<year>.tif", "path/to/outline_<year>.geojson"),
    2015: ("dem_2015.tif", "outline_2015.geojson"),
    2016: ("dem_2016.tif", "outline_2015.geojson"),  # reusing 2015 outline
    # ... 
}

# Buffered outline - used to build the mesh, it is union of all years outlines
BUFFER_OUTLINE = "outline_buffered.geojson"

# Output checkpoint file — this is what your flow model (C2_iceVelocity) will load
OUTPUT_FILE = "bedrock_mean.h5"

# DEM values below this threshold are treated as NoData and filled
# using nearest-neighbour interpolation before smoothing.
# Set this safely below your glacier's minimum surface elevation.
NODATA_THRESHOLD = 1000.0

# Physical constants 
rho   = 917.0       # Ice density [kg/m³]
g     = 9.81        # Gravitational acceleration [m/s²]
tau_c = 115000.0    # Plastic yield stress [Pa], typical range 50–150 kPa for valley glaciers

# Tikhonov smoothing parameters
alpha_s = firedrake.Constant(60.0)   # Surface smoothing length scale [m], suppresses DEM noise
alpha_h = firedrake.Constant(90.0)   # Thickness smoothing length scale [m], controls bedrock smoothness
penalty = firedrake.Constant(1e6)    # Edge penalty weight — forces ice thickness to zero outside the glacier outline


# ==============================================================
# 1. BUILD MESH
# The mesh is built from the buffered outline and reused for all
# years. 
# ==============================================================
print("=" * 60)
print("STEP 1: BUILDING MESH")
print("=" * 60)

with open(BUFFER_OUTLINE, 'r') as f:
    outline_json = geojson.load(f)

outline_clean = geojson.utils.map_tuples(lambda x: x[:2], outline_json)
geometry      = icepack.meshing.collection_to_gmsh(outline_clean)
geometry.write("glacier_mesh.msh")
mesh = firedrake.Mesh("glacier_mesh.msh")

print(f"  > Mesh created: {mesh.num_cells()} triangles")

Q = firedrake.FunctionSpace(mesh, "CG", 2)
V = firedrake.VectorFunctionSpace(mesh, "CG", 2)

# Extract node coordinates for NoData filling
# point-in-polygon mask tests
coords_func = firedrake.Function(V).interpolate(firedrake.SpatialCoordinate(mesh))
coords_all  = coords_func.dat.data_ro   # shape (N, 2)
assert coords_all.shape[1] == 2, (
    f"Expected 2D node coordinates, got shape {coords_all.shape}"
)
n_nodes = len(coords_all)
n_years = len(YEARS)


# ==============================================================
# 2. LOAD EACH YEAR — smooth DEM, build mask, accumulate
# ==============================================================
print()
print("=" * 60)
print("STEP 2: LOADING AND SMOOTHING EACH YEAR'S DEM")
print("=" * 60)

# Each node's mean surface is computed only over years in which
# that node was inside the glacier outline. This prevents mixing
# ice-surface elevations (early years) with bare-rock elevations


s_sum_inside  = np.zeros(n_nodes)   # sum of surface elevation, glacier-covered years only
counts_inside = np.zeros(n_nodes)   # number of years each node was inside the glacier
s_sum_all     = np.zeros(n_nodes)   # sum of surface elevation over all years 
k_union       = np.zeros(n_nodes)   # union glacier mask, updated each year

for year, (dem_file, mask_file) in YEARS.items():
    print(f"\n  --- Year {year} ---")

    # Load DEM and fill NoData holes into a separate function (s_filled)
    # so s_raw remains the original raster interpolation and is not mutated.
    with rasterio.open(dem_file) as dsm:
        s_raw = icepack.interpolate(dsm, Q)
    s_filled = firedrake.Function(Q, name=f"s_filled_{year}").assign(s_raw)
    s_data   = s_filled.dat.data   # writable reference into s_filled

    bad  = s_data < NODATA_THRESHOLD
    good = ~bad
    if np.any(bad):
        n_bad = int(bad.sum())
        print(f"  > Filling {n_bad} NoData nodes with nearest neighbour")
        interp      = NearestNDInterpolator(coords_all[good], s_data[good])
        s_data[bad] = interp(coords_all[bad])

    # Tikhonov smoothing (J_s): find s_yr that minimises
    #   0.5*(s_yr - s_filled)^2 + 0.5*alpha_s^2*|grad s_yr|^2
    s_yr = firedrake.Function(Q, name=f"s_{year}").assign(s_filled)
    J_s  = (
        0.5 * (s_yr - s_filled)**2 * firedrake.dx
        + 0.5 * alpha_s**2
          * firedrake.inner(firedrake.grad(s_yr), firedrake.grad(s_yr))
          * firedrake.dx
    )
    firedrake.solve(firedrake.derivative(J_s, s_yr) == 0, s_yr)

    s_yr_vals = s_yr.dat.data_ro.copy()
    print(f"  > Surface smoothed. Range: {s_yr_vals.min():.1f} – {s_yr_vals.max():.1f} m")

    # Build glacier mask from the outline for this year.
    
    with open(mask_file, 'r') as f:
        mask_json = geojson.load(f)
    mask_poly = unary_union([shape(feat['geometry']) for feat in mask_json['features']])

    k_yr = np.array(
        [1.0 if mask_poly.contains(Point(x, y)) else 0.0 for x, y in coords_all],
        dtype=float
    )
    print(f"  > Mask built: {int(k_yr.sum())} nodes inside glacier")

    # Accumulate sums for the mean surface calculation.
    s_sum_all     += s_yr_vals
    s_sum_inside  += s_yr_vals * k_yr
    counts_inside += k_yr
    k_union        = np.maximum(k_union, k_yr)


# ==============================================================
# 3. COMPUTE MEAN SURFACE
# ==============================================================
print()
print("=" * 60)
print("STEP 3: COMPUTING MEAN SURFACE")
print("=" * 60)

counts_safe = np.maximum(counts_inside, 1)   # prevent division by zero for buffer-zone nodes

s_mean_data = np.where(
    counts_inside > 0,
    s_sum_inside / counts_safe,   # glacier nodes: average over ice-covered years only
    s_sum_all    / n_years        # buffer zone:   average over all years (bare rock, not used in physics)
)

s_mean = firedrake.Function(Q, name="Mean_Surface")
s_mean.dat.data[:] = s_mean_data


# ==============================================================
# 4. NYE BEDROCK FROM MEAN SURFACE
# ==============================================================

# Build the union mask — covers every node that was inside the
# glacier in at least one survey year. The Nye bedrock is computed
# and tapered over this full area. When the flow model runs for a
# specific year, that year's own outline mask will zero out
# thickness where the glacier no longer exists.
k_mean = firedrake.Function(Q, name="Union_Mask")
k_mean.dat.data[:] = k_union

print()
print("=" * 60)
print("STEP 4: NYE BEDROCK FROM MEAN SURFACE")
print("=" * 60)

# Surface slope magnitude.
slope_mag = (
    firedrake.sqrt(firedrake.inner(firedrake.grad(s_mean), firedrake.grad(s_mean)))
    + firedrake.Constant(1e-5)
)

# Nye (1952) thickness:  h = tau_c / (rho * g * |grad s|)
h_expr = firedrake.Constant(tau_c) / (
    firedrake.Constant(rho) * firedrake.Constant(g) * slope_mag
)
h_raw = firedrake.Function(Q).interpolate(h_expr)

# Clamp thickness to a physically plausible range and apply
# the union mask. The 400 m upper bound prevents unrealistically
# large values on near-flat terrain, adjust if your glacier is
# known to be deeper.
h_clamped = firedrake.Function(Q).interpolate(
    firedrake.min_value(400.0, firedrake.max_value(0.0, h_raw))
)
h_clamped.interpolate(h_clamped * k_mean)

raw_inside = h_clamped.dat.data_ro[k_union > 0.5]
print(f"  > Raw clamped h (union mask): "
      f"min={raw_inside.min():.1f}  "
      f"mean={raw_inside.mean():.1f}  "
      f"max={raw_inside.max():.1f} m")

# Tikhonov smoothing (J_h) with edge penalty.

h_mean = firedrake.Function(Q, name="Mean_Thickness").assign(h_clamped)

J_h = (
    0.5 * (h_mean - h_clamped)**2 * firedrake.dx
    + 0.5 * alpha_h**2
      * firedrake.inner(firedrake.grad(h_mean), firedrake.grad(h_mean))
      * firedrake.dx
    + 0.5 * penalty
      * firedrake.max_value(0.0, 1.0 - k_mean)
      * (h_mean**2)
      * firedrake.dx
)
firedrake.solve(firedrake.derivative(J_h, h_mean) == 0, h_mean)
print("  > J_h smoothing solved.")

# Reapply mask and enforce non-negativity after the solve,
# since the PDE solution may produce small spurious values outside
# the glacier boundary.
h_mean.interpolate(h_mean * k_mean)
h_mean.interpolate(firedrake.max_value(0.0, h_mean))

smooth_inside = h_mean.dat.data_ro[k_union > 0.5]
print(f"  > Smoothed h (union mask): "
      f"min={smooth_inside.min():.1f}  "
      f"mean={smooth_inside.mean():.1f}  "
      f"max={smooth_inside.max():.1f} m")

# Bedrock elevation = mean surface - smoothed mean thickness.
b_mean = firedrake.Function(Q, name="Master_Bed_Topography")
b_mean.interpolate(s_mean - h_mean)

b_inside = b_mean.dat.data_ro[k_union > 0.5]
print(f"  > Mean bedrock (union mask): "
      f"min={b_inside.min():.1f}  "
      f"mean={b_inside.mean():.1f}  "
      f"max={b_inside.max():.1f} m")


# ==============================================================
# 5. SAVE TO CHECKPOINT
# ==============================================================
print()
print("=" * 60)
print(f"STEP 5: SAVING TO {OUTPUT_FILE}")
print("=" * 60)

with firedrake.CheckpointFile(OUTPUT_FILE, 'w') as chk:
    chk.save_mesh(mesh)
    chk.save_function(b_mean, name="Master_Bed_Topography")

print(f"  > Saved: {OUTPUT_FILE}")
print()
print("=" * 60)
print("DONE")
print("=" * 60)


In [ ]:
# ==============================================================
# 6. VISUALISATION — Mean Surface, Thickness, Bedrock
# ==============================================================
import matplotlib.pyplot as plt

print()
print("=" * 60)
print("STEP 6: GENERATING MAPS")
print("=" * 60)

# ── Numeric statistics printed to console ─────────────────────
inside = k_union > 0.5

s_in = s_mean.dat.data_ro[inside]
h_in = h_mean.dat.data_ro[inside]
b_in = b_mean.dat.data_ro[inside]

print()
print("  Mean Surface (glacier nodes):")
print(f"    min={s_in.min():.1f}  mean={s_in.mean():.1f}  max={s_in.max():.1f} m")
print()
print("  Mean Thickness (glacier nodes):")
print(f"    min={h_in.min():.1f}  mean={h_in.mean():.1f}  max={h_in.max():.1f} m")
print()
print("  Mean Bedrock (glacier nodes):")
print(f"    min={b_in.min():.1f}  mean={b_in.mean():.1f}  max={b_in.max():.1f} m")
print()
print("  Consistency checks:")
assert np.all(b_in <= s_in), f"FAIL: bedrock above surface at {(b_in > s_in).sum()} nodes"
assert np.all(h_in >= 0),    f"FAIL: negative thickness at {(h_in < 0).sum()} nodes"
print("    bedrock <= surface : PASS")
print("    thickness >= 0     : PASS")



# ── 1×3 map figure ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Mean Bedrock Calculation — Diagnostic Maps", fontsize=16, fontweight="bold")

# Mean surface elevation
map_s = firedrake.tripcolor(s_mean, axes=axes[0], cmap="terrain")
axes[0].set_title("Mean Surface Elevation (s_mean) [m]", fontsize=12)
fig.colorbar(map_s, ax=axes[0], label="Elevation (m)")
axes[0].set_aspect("equal")

# (Mean) ice thickness
map_h = firedrake.tripcolor(h_mean, axes=axes[1], cmap="nipy_spectral")
axes[1].set_title("Mean Ice Thickness (h_mean) [m]", fontsize=12)
fig.colorbar(map_h, ax=axes[1], label="Thickness (m)")
axes[1].set_aspect("equal")

# (Mean) bedrock elevation
map_b = firedrake.tripcolor(b_mean, axes=axes[2], cmap="terrain")
axes[2].set_title("Mean Bedrock Elevation (b_mean) [m]", fontsize=12)
fig.colorbar(map_b, ax=axes[2], label="Elevation (m)")
axes[2].set_aspect("equal")

plt.tight_layout()

output_map = "bedrock_diagnostic.png"
plt.savefig(output_map, dpi=200, bbox_inches="tight")
print(f"  > Map saved as: {output_map}")
plt.show()
